In [1]:
# Import chunk
import pandas as pd

## 1. Load and Prepare Data

In [2]:
# Load isco data
isco = pd.read_excel(
    "../data/raw/ISCO-08 EN Structure and definitions.xlsx",
    dtype={"ISCO 08 Code": str} # necessary to keep leading zeros
    )
isco.head()

,Level,ISCO 08 Code,Title EN,Definition,Tasks include,Included occupations,Excluded occupations,Notes
0,4,1114,Senior Officials of Special-interest Organizat...,Senior officials of special-interest organizat...,Tasks include -\n(a) determining and formulat...,Examples of the occupations classified here:\n...,NaN,NaN
1,4,1212,Human Resource Managers,"Human resource managers, plan, direct and coor...","Tasks include -\n(a) planning, directing and ...",Examples of the occupations classified here:\n...,NaN,NaN
2,1,1,Managers,"Managers plan, direct, coordinate and evaluate...",Tasks performed by managers usually include: f...,Occupations in this major group are classified...,NaN,In distinguishing between managers classified ...
3,1,2,Professionals,Professionals increase the existing stock of k...,Tasks performed by professionals usually inclu...,Occupations in this major group are classified...,NaN,NaN
4,1,3,Technicians and Associate Professionals,Technicians and associate professionals perfor...,Tasks performed by technicians and associate p...,Occupations in this major group are classified...,NaN,NaN


In [3]:
# Rename columns
isco.rename(
    columns={"Level":"level",
             "ISCO 08 Code":"isco_code",
             "Title EN":"title",
             "Definition":"definition",
             "Tasks include":"tasks"},
             inplace=True
             )

# Drop unnecessary columns
isco.drop(
    columns=['Included occupations', 'Excluded occupations', 'Notes'], 
    inplace=True
    )

# Drop level 4 data
isco3 = isco[isco["level"] <= 3].copy()

# Check for rows with NAs
display(isco3[isco3.isna().any(axis=1)])

# Fill NAs with empty strings
isco3.fillna("", inplace=True)

,level,isco_code,title,definition,tasks
11,1,0,Armed Forces Occupations,Armed forces occupations include all jobs held...,NaN
52,2,01,Commissioned Armed forces Officers,Commissioned armed forces officers provide lea...,NaN
53,2,02,Non-commissioned Armed Forces Officers,Non-commissioned armed forces officers enforce...,NaN
54,2,03,"Armed Forces Occupations, Other Ranks","Armed forces occupations, other ranks include ...",NaN
182,3,011,Commissioned Armed Forces Officers,Commissioned armed forces officers provide lea...,NaN
183,3,021,Non-commissioned Armed Forces Officers,Non-commissioned armed forces officers enforce...,NaN
184,3,031,"Armed Forces Occupations, Other Ranks","Armed forces occupations, other ranks include ...",NaN


## 2. Create Prompt Texts

In [4]:
# Create mappings from code to title
major_map = (
    isco3[isco3["level"] == 1]
    .set_index("isco_code")["title"]
    .to_dict()
)

submajor_map = (
    isco3[isco3["level"] == 2]
    .set_index("isco_code")["title"]
    .to_dict()
)

# Create hierarchy codes
isco3["major_code"] = isco3["isco_code"].str[:1]
isco3["submajor_code"] = isco3["isco_code"].str[:2]

# Create hierarchy titles
isco3["major_group"] = isco3["major_code"].map(major_map)
isco3["submajor_group"] = isco3["submajor_code"].map(submajor_map)

In [5]:
# Create datasets for only level 2 and level 3 descriptions
isco_lvl2 = isco3[isco3["level"] == 2].copy().reset_index(drop=True)
isco_lvl3 = isco3[isco3["level"] == 3].copy().reset_index(drop=True)

In [6]:
# Create prompt texts
isco_lvl2["prompt_text"] = (
    "Major group:\n" + isco_lvl2["major_group"] +
    "\n\nSub-major group:\n" + isco_lvl2["title"] +
    "\n\nDefinition:\n" + isco_lvl2["definition"] +
    "\n\nTasks include:\n" + isco_lvl2["tasks"]
)

isco_lvl3["prompt_text"] = (
    "Major group:\n" + isco_lvl3["major_group"] +
    "\n\nSub-major group:\n" + isco_lvl3["submajor_group"] +
    "\n\nMinor group:\n" + isco_lvl3["title"] +
    "\n\nDefinition:\n" + isco_lvl3["definition"] +
    "\n\nTasks include:\n" + isco_lvl3["tasks"]
)

# Examples
print("=== Example level 2 === \n\n",isco_lvl2["prompt_text"].iloc[0], "\n\n")
print("=== Example level 3 === \n\n",isco_lvl3["prompt_text"].iloc[0])

=== Example level 2 === 

 Major group:
Managers

Sub-major group:
Chief Executives, Senior Officials and Legislators

Definition:
Chief executives, senior officials and legislators formulate and review the policies, and plan, direct, coordinate and evaluate the overall activities, of enterprises, governments and other organizations with the support of other managers. Competent performance in most occupations in this sub-major group requires skills at the fourth ISCO skill level.

Tasks include:
Tasks performed by workers in this sub-major group usually include: presiding over or participating in the proceedings of legislative bodies, boards of directors and committees; formulating and advising on the policy budgets, laws and regulations of enterprises, governments and other organizations; establishing objectives for enterprises, government departments or agencies and other organizations; formulating or approving and evaluating programmes and policies and  procedures for their implemen

## 3. Export Data

In [7]:
# Drop unnecessary columns
isco_lvl2.drop(
    columns=['level','submajor_code','submajor_group'], # title and isco_code already contain this information
    inplace=True
    )

isco_lvl3.drop(
    columns=['level'], 
    inplace=True
    )

In [8]:
# Export data 
isco_lvl2.to_csv("../data/isco_level2.csv", index=False)
isco_lvl3.to_csv("../data/isco_level3.csv", index=False)

# Test
test = pd.read_csv("../data/isco_level2.csv")
print(test.iloc[0])

isco_code                                                     11
title          Chief Executives, Senior Officials and Legisla...
definition     Chief executives, senior officials and legisla...
tasks          Tasks performed by workers in this sub-major g...
major_code                                                     1
major_group                                             Managers
prompt_text    Major group:\nManagers\n\nSub-major group:\nCh...
Name: 0, dtype: object
